# Week 1: Python 数据工具与"数据是什么"

## 学习目标

1. 理解 observation（观测）、feature（特征）、target（目标）和 unit（单位）
2. 掌握 NumPy 和 Pandas 基础操作
3. 学会使用 Matplotlib 进行数据可视化
4. 完成 EDA（探索性数据分析）流程

## 1. 数据读取与检查

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# 设置绘图参数
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 12

# 读取数据
df = pd.read_csv('data/sample_bike_data.csv')

# 查看基本信息
print("数据形状:", df.shape)
print("\n列名:", df.columns.tolist())
print("\n数据类型:")
print(df.dtypes)

In [ ]:
# 查看前几行
df.head(10)

In [ ]:
# 描述性统计
df.describe()

## 2. 理解数据结构

### 关键概念

- **observation（观测）**：每一行代表什么？
- **feature（特征）**：输入变量有哪些？
- **target（目标）**：我们要预测什么？
- **unit（单位）**：数值的单位是什么？

In [ ]:
# 回答关键问题
print("每一行代表：某站点某小时的记录")
print("\n特征列：")
features = ['hour', 'available_bikes', 'temperature', 'is_weekend', 'is_holiday']
print(features)
print("\n目标列：demand（需求量，单位：人次/小时）")
print("\n各列单位：")
units = {
    'station_id': '站点编号',
    'hour': '小时 (0-23)',
    'demand': '人次/小时',
    'available_bikes': '辆',
    'temperature': '摄氏度',
    'is_weekend': '布尔值 (0/1)',
    'is_holiday': '布尔值 (0/1)'
}
for col, unit in units.items():
    print(f"  {col}: {unit}")

## 3. 数据清洗

In [ ]:
# 检查缺失值
print("缺失值统计:")
print(df.isnull().sum())

# 检查重复行
print(f"\n重复行数: {df.duplicated().sum()}")

In [ ]:
# 检查异常值
print("需求量范围:", df['demand'].min(), "-", df['demand'].max())
print("可用车辆范围:", df['available_bikes'].min(), "-", df['available_bikes'].max())
print("温度范围:", df['temperature'].min(), "-", df['temperature'].max())

In [ ]:
# 检查负值（如果存在需要处理）
negative_demand = df[df['demand'] < 0]
if len(negative_demand) > 0:
    print(f"发现 {len(negative_demand)} 条负需求记录，需要处理")
    df.loc[df['demand'] < 0, 'demand'] = 0
else:
    print("没有发现负需求记录")

## 4. 探索性数据分析（EDA）

In [ ]:
# 按站点分组统计
station_stats = df.groupby('station_id')['demand'].agg(['mean', 'std', 'min', 'max', 'sum'])
print("各站点需求统计:")
print(station_stats)

In [ ]:
# 按小时分组
hourly_demand = df.groupby('hour')['demand'].mean()
print("每小时平均需求:")
print(hourly_demand)

In [ ]:
# 工作日 vs 周末
weekend_compare = df.groupby('is_weekend')['demand'].mean()
print("工作日 vs 周末平均需求:")
print(f"工作日: {weekend_compare[0]:.2f}")
print(f"周末: {weekend_compare[1]:.2f}")

## 5. 数据可视化

In [ ]:
# 创建多子图
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# 图1: 需求分布直方图
axes[0, 0].hist(df['demand'], bins=30, color='#2196F3', alpha=0.7, edgecolor='white')
axes[0, 0].set_title('需求分布')
axes[0, 0].set_xlabel('需求量')
axes[0, 0].set_ylabel('频数')
axes[0, 0].axvline(df['demand'].mean(), color='red', linestyle='--', label=f'均值: {df["demand"].mean():.1f}')
axes[0, 0].legend()

# 图2: 每小时平均需求
axes[0, 1].plot(hourly_demand.index, hourly_demand.values, marker='o', linewidth=2, color='#4CAF50')
axes[0, 1].set_title('每小时平均需求')
axes[0, 1].set_xlabel('小时')
axes[0, 1].set_ylabel('需求量')
axes[0, 1].set_xticks(range(0, 24, 2))
axes[0, 1].grid(True, alpha=0.3)

# 图3: 各站点总需求
station_total = df.groupby('station_id')['demand'].sum()
axes[1, 0].bar(station_total.index, station_total.values, color=['#FF9800', '#9C27B0', '#E91E63'])
axes[1, 0].set_title('各站点总需求')
axes[1, 0].set_xlabel('站点')
axes[1, 0].set_ylabel('总需求')

# 图4: 需求 vs 温度
axes[1, 1].scatter(df['temperature'], df['demand'], alpha=0.5, color='#00BCD4')
axes[1, 1].set_title('需求 vs 温度')
axes[1, 1].set_xlabel('温度 (°C)')
axes[1, 1].set_ylabel('需求量')

plt.tight_layout()
plt.savefig('eda_overview.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# 多站点需求曲线对比
fig, ax = plt.subplots(figsize=(12, 6))

colors = {'A001': '#2196F3', 'A002': '#4CAF50', 'A003': '#FF9800'}

for station in df['station_id'].unique():
    station_data = df[df['station_id'] == station]
    hourly = station_data.groupby('hour')['demand'].mean()
    ax.plot(hourly.index, hourly.values, marker='o', label=station, 
            color=colors[station], linewidth=2, markersize=6)

ax.set_title('各站点 24 小时需求曲线')
ax.set_xlabel('小时')
ax.set_ylabel('平均需求')
ax.set_xticks(range(0, 24, 2))
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 6. Research Thinking

### 问题 1：数据生成过程

数据表中的每一行代表什么现实过程？数据质量问题会怎样改变研究结论？

**回答：**

每一行代表一个站点在特定小时内的需求记录。这个需求是"有多少人想租借单车"的统计。

数据质量问题的影响：
1. 如果某些站点数据经常缺失，会低估该站点的实际需求
2. 如果周末数据缺失较多，会高估工作日的需求
3. 如果高需求时段记录丢失，会导致对峰值需求的低估

### 问题 2：观察 vs 干预

仅凭观察数据可能得出错误结论的例子：

**例子：** 我们可能发现"温度越高，需求越大"，但这不意味着只要调高温度计读数就能增加需求。

真正的原因可能是：
- 温度高时是白天，活动人多
- 温度高时可能是周末或假期
- 真正影响需求的是"人们是否出门活动"，温度只是关联变量

### 问题 3：单位检查

为什么单位检查是数据分析的第一步？

**回答：**

单位错误可能导致：
1. 数值比较无意义（例如比较"人次"和"人天"）
2. 公式计算错误（例如需求率 = 需求 / 可用车辆，单位应该是"人次/辆"）
3. 决策失误（例如把"每小时的需求数量"当成"全天的需求数量"来调配车辆）

**实际案例：** 如果温度数据误用了华氏度而非摄氏度，可能会在 25°C 时以为是 25°F（约 -4°C），错误地判断为低温天气。

## 7. 总结

### 本周学习要点

1. **数据结构**：理解 observation、feature、target、unit
2. **Pandas 操作**：读取、选择、过滤、分组、聚合
3. **NumPy 基础**：数组操作、向量化计算
4. **Matplotlib 可视化**：折线图、柱状图、散点图、直方图
5. **EDA 流程**：数据检查 → 清洗 → 分析 → 可视化 → 解释

### 关键洞察

- A001（商业区）早晚高峰明显
- A002（住宅区）早上流出多，晚上流入多
- A003（景区）白天活跃，夜间很低
- 工作日需求高于周末
- 温度与需求正相关

In [ ]:
# 保存清洗后的数据
df.to_csv('data/cleaned_bike_data.csv', index=False)
print("数据已保存到 data/cleaned_bike_data.csv")